# Project

## Loading the dataset

In [ ]:
import glob
from typing import Tuple
import polars as pl
import pandas as pd
import os

In [ ]:
DATA_ROOT = "data"

In [ ]:
def get_labels_from_filename(filename: str) -> Tuple[str, str, bool]:
    if "Benign" in filename:
        return "Benign", "Benign", False

    if "ARP_Spoofing" in filename:
        return "ARP_Spoofing", "Spoofing", True

    if "Recon" in filename:
        category = "Recon"
        if "Recon-Ping_Sweep" in filename:
            return "Ping_Sweep", category, True
        if "Recon-VulScan" in filename:
            return "VulScan", category, True
        if "Recon-OS_Scan" in filename:
            return "OS_Scan", category, True
        if "Recon-Port_Scan" in filename:
            return "Port_Scan", category, True

    if "MQTT" in filename:
        category = "MQTT"
        if "MQTT-Malformed_Data" in filename:
            return "Malformed_Data", category, True
        if "MQTT-DoS-Connect_Flood" in filename:
            return "DoS_Connect_Flood", category, True
        if "MQTT-DDoS-Publish_Flood" in filename:
            return "DDoS_Publish_Flood", category, True
        if "MQTT-DoS-Publish_Flood" in filename:
            return "DoS_Publish Flood", category, True
        if "MQTT-DDoS-Connect_Flood" in filename:
            return "DDoS_Connect_Flood", category, True

    if "TCP_IP-DoS" in filename:
        category = "DoS"
        if "TCP_IP-DoS-TCP" in filename:
            return "DoS_TCP", category, True
        if "TCP_IP-DoS-ICMP" in filename:
            return "DoS_ICMP", category, True
        if "TCP_IP-DoS-SYN" in filename:
            return "DoS_SYN", category, True
        if "TCP_IP-DoS-UDP" in filename:
            return "DoS_UDP", category, True

    if "TCP_IP-DDoS" in filename:
        category = "DDoS"
        if "TCP_IP-DDoS-SYN" in filename:
            return "DDoS_SYN", category, True
        if "TCP_IP-DDoS-TCP" in filename:
            return "DDoS_TCP", category, True
        if "TCP_IP-DDoS-ICMP" in filename:
            return "DDoS_ICMP", category, True
        if "TCP_IP-DDoS-UDP" in filename:
            return "DDoS_UDP", category, True

    return "", "", False 

In [ ]:
def combine_dataset_files(root: str, save_name: str = "combined_dataset") -> None:
    path = os.path.join(root,"*","*.csv")
    files = glob.glob(path, recursive=True)
    combined_dir = os.path.join(root, "combined")
    out_path = os.path.join(combined_dir, f"{save_name}.parquet")
    os.makedirs(combined_dir, exist_ok=True)

    frames = []
    with pl.StringCache():
        for file in files:
            name, category, attack = get_labels_from_filename(file)

            lf = pl.scan_csv(file)

            lf = lf.with_columns(
                [
                    pl.lit(name).alias("Name").cast(pl.Categorical),
                    pl.lit(category).alias("Category").cast(pl.Categorical),
                    pl.lit(attack).alias("Attack"),
                ]
            )

            frames.append(lf)

        combined_lf = pl.concat(frames)
        combined_lf.sink_parquet(out_path)

In [ ]:
def load_dataset(save_name: str) -> pl.DataFrame:
    path = os.path.join(DATA_ROOT, "combined", f"{save_name}.parquet")
    if not os.path.exists(path):
        combine_dataset_files(DATA_ROOT, save_name)
    df = pl.read_parquet(path)
    return df

In [ ]:
df = load_dataset("combined_dataset").with_columns(pl.col(pl.Float64).cast(pl.Float32)) # reducing memory usage

## Exploratory Data Analysis

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
sns.set_theme(style="darkgrid", context="paper", palette="bright")

In [ ]:
def plot_attack_counts(data):
    attack_counts = data.group_by("Name").len().sort("len", descending=False)
    attack_counts = attack_counts.to_pandas()
    fig, ax = plt.subplots(figsize = (12,6))
    sns.barplot(x="Name", y="len", data= attack_counts, ax=ax)
    ax.set_title("Distribution of Attack Names (Entire Dataset)")
    ax.set_xlabel("Attack Name")
    ax.set_ylabel("Count")

    ax.set_xticklabels(
        ax.get_xticklabels(),
        rotation = 45,
        horizontalalignment="right",
        rotation_mode = "anchor"
    )

    plt.ticklabel_format(style = "plain", axis = "y")
    plt.tight_layout()

In [ ]:
plot_attack_counts(df)

In [ ]:
def plot_ddos_dos_bar(data):
    ddos_dos_counts = (
        data.filter(pl.col("Category").is_in(["DDoS", "DoS"]))
        .group_by("Category")
        .len()
        .sort("len", descending=False)
    )
    data = ddos_dos_counts.to_pandas()
    data.columns = ["Attack Type", "Count"]

    fig, ax = plt.subplots(figsize = (12,6))
    sns.barplot(x="Attack Type", y="Count", data=data, ax=ax)
    ax.set_title("Distribution of DDoS and DoS Attacks (Entire Dataset)")
    ax.set_xlabel("Attack Category")
    ax.set_ylabel("Count")

    ax.set_xticklabels(
        ax.get_xticklabels(),
        rotation = 45,
        horizontalalignment="right",
        rotation_mode = "anchor"
    )

    plt.ticklabel_format(style = "plain", axis = "y")
    plt.tight_layout()

In [ ]:
plot_ddos_dos_bar(df)

In [ ]:
def plot_categories_no_ddos_dos(data):
    included_categories = ["MQTT", "Spoofing", "Recon", "Benign"]
    data = (
        data.filter(pl.col("Category").is_in(included_categories))
        .group_by("Category")
        .len()
        .sort("len", descending=False)
    ).to_pandas()

    fig, ax = plt.subplots(figsize = (12,6))
    sns.barplot(x="Category", y="len", data=data, ax=ax)
    ax.set_title("Distribution of Attack Categories (Excluding DDoS and DoS) (Entire Dataset)")
    ax.set_xlabel("Attack Category")
    ax.set_ylabel("Count")

    ax.set_xticklabels(
        ax.get_xticklabels(),
        rotation = 45,
        horizontalalignment="right",
        rotation_mode = "anchor"
    )

    plt.ticklabel_format(style = "plain", axis = "y")
    plt.tight_layout()

In [ ]:
plot_categories_no_ddos_dos(df)

In [ ]:
def plot_all_attack_categories_bar(data):
    data = data.group_by("Category").len().sort("len", descending=False).to_pandas()
    fig, ax = plt.subplots(figsize = (12,6))
    sns.barplot(x="Category", y="len", data=data, ax=ax)
    ax.set_title("Distribution of Attack Categories (Entire Dataset)")
    ax.set_xlabel("Attack Category")
    ax.set_ylabel("Count")

    ax.set_xticklabels(
        ax.get_xticklabels(),
        rotation = 45,
        horizontalalignment="right",
        rotation_mode = "anchor"
    )

    plt.ticklabel_format(style = "plain", axis = "y")
    plt.tight_layout()

In [ ]:
plot_all_attack_categories_bar(df)

In [ ]:
def plot_attack_benign_pie(data):
    attack_counts = data.group_by("Attack").len().sort("len", descending=False)
    attack_counts = attack_counts.to_pandas()
    attack_counts.columns = ["Attack", "Count"]

    attack_counts = attack_counts.set_index("Attack")["Count"]
    total_count = attack_counts.sum()
    percentages = (attack_counts / total_count) * 100
    slices = attack_counts.values
    labels = attack_counts.index

    colours = sns.color_palette("hls", len(labels))

    fig, ax = plt.subplots()
    ax.set_title("Comparison between Benign and Attack Data (Entire Dataset)")
    plt.pie(
        slices,
        colors=colours,  
        wedgeprops=dict(edgecolor="black", linewidth=1),
        textprops=dict(color="white", fontsize=10),
        startangle=90,
    )

    legend_labels = [
        f"{label} ({percentage:.2f}%)"
        for label, percentage in zip(labels, percentages)
    ]
    ax.legend(
        legend_labels,
        title="Malicious Traffic?",
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
    )

    plt.gca().set_aspect("equal")

In [ ]:
plot_attack_benign_pie(df)

In [ ]:
def plot_breakdown_pie(data, title):
    target_column = "Category"
    attack_counts = data.group_by(target_column).len().sort(target_column)
    attack_counts = attack_counts.to_pandas()
    attack_counts.columns = [target_column, "Count"]

    attack_counts = attack_counts.set_index(target_column)["Count"]
    total_count = attack_counts.sum()
    percentages = (attack_counts / total_count) * 100
    slices = attack_counts.values
    labels = attack_counts.index

    fig, ax = plt.subplots()
    ax.set_title(title)
    colours = sns.color_palette("tab20", len(labels))
    plt.pie(
        slices,
        wedgeprops=dict(edgecolor="black", linewidth=1),
        textprops=dict(color="white", fontsize=10),
        startangle=90,
        colors=colours,
    )

    legend_labels = [
        f"{label} ({percentage:.2f}%)"
        for label, percentage in zip(labels, percentages)
    ]
    ax.legend(
        legend_labels,
        title=f"Attack {target_column}",
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
    )

    plt.gca().set_aspect("equal")

In [ ]:
def plot_correlation_heatmap(data):
    numeric_data = data.select(
        pl.exclude("Attack", "Category", "Name").name.map(lambda name: name.lower())
    )

    std_devs = numeric_data.select(pl.all().std())
    zer_var_cols = [col for col in std_devs.columns if std_devs[col][0] == 0]
    numeric_data = numeric_data.drop(zer_var_cols)

    corr = numeric_data.corr()
    corr_labels = corr.columns
    corr = corr.to_numpy()
    mask = np.triu(np.ones_like(corr, dtype=bool))

    fig, ax = plt.subplots(figsize=(15, len(corr_labels) // 2))
    sns.heatmap(
        corr,
        cmap="coolwarm",
        square=True,
        ax=ax,
        mask=mask,
        cbar_kws={"label":"Correlation Coefficient","shrink":0.5,"aspect":30},
        xticklabels=corr_labels,
        yticklabels=corr_labels,
    )

In [ ]:
plot_correlation_heatmap(df)

## Preprocessing

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

In [ ]:
df = df.to_pandas()

In [ ]:
df.columns

In [ ]:
X = df.drop(columns = ["Name", "Attack", "Category"])
y = df["Category"]

### Splitting Data into Subsets

In [ ]:
train_percent = 0.8
val_percent = 0.1
test_percent = 0.1

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=test_percent, random_state=0, stratify=y
)

val_adjusted_size = val_percent / (train_percent + val_percent)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=val_adjusted_size, random_state=0, stratify=y_train_val
)

In [ ]:
print(f"Train {len(X_train)/len(X):.0%} ({len(X_train)})")
print(f"Validation {len(X_val)/len(X):.0%} ({len(X_val)})")
print(f"Test {len(X_test)/len(X):.0%} ({len(X_test)})")

### Random Undersampling

In [ ]:
# plot_breakdown_pie(pl.DataFrame({"Category": y_train}), "Training Set Before Undersampling")
# plot_breakdown_pie(pl.DataFrame({"Category": y_val}), "Validation Set Before Undersampling")
# plot_breakdown_pie(pl.DataFrame({"Category": y_test}), "Test Set Before Undersampling")

In [ ]:
# rus = RandomUnderSampler(sampling_strategy={"DDoS": 300_000, "DoS": 300_000}, random_state=0)
# X_train, y_train = rus.fit_resample(X_train, y_train)

In [ ]:
# plot_breakdown_pie(pl.DataFrame({"Category": y_train}), "Training Set After Undersampling")
# plot_breakdown_pie(pl.DataFrame({"Category": y_val}), "Validation Set After Undersampling")
# plot_breakdown_pie(pl.DataFrame({"Category": y_test}), "Testing Set After Undersampling")

### Scaling and Labelling Data

In [ ]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_val = encoder.transform(y_val)
y_test = encoder.transform(y_test)

In [ ]:
scaler = StandardScaler()
scaler.set_output(transform="pandas")
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

### Feature Selection

In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression

In [ ]:
l1_model = LogisticRegression(
    penalty = "l1",
    solver = "saga",
    C=0.01,
    multi_class="multinomial",
    max_iter=5000
)
l1_model.fit(X_train_scaled, y_train)

In [ ]:
selector = SelectFromModel(l1_model, prefit=True)
X_train_scaled = selector.transform(X_train_scaled)
X_val_scaled = selector.transform(X_val_scaled)

In [ ]:
feature_mask = selector.get_support()
dropped_features = np.array(X_train.columns.tolist())[~feature_mask]
kept_features = np.array(X_train.columns.tolist())[feature_mask]

In [ ]:
print(f"Kept features: {kept_features}")
print(f"Dropped features: {dropped_features}")

In [ ]:
print(f"Training shape: {X_train_scaled.shape}")
print(f"Validation shape: {X_val_scaled.shape}")
print(f"Testing shape: {X_test_scaled.shape}")

## Model Design

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, BatchNormalization, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam

In [ ]:
def build_dnn(input_dimensions):
    model = Sequential([
        Input(shape=(input_dimensions[1],)),

        Dense(256, activation="relu"),
        BatchNormalization(),
        Dropout(0.3),

        Dense(128, activation="relu"),
        BatchNormalization(),
        Dropout(0.2),

        Dense(64, activation="relu"),
        
        Dense(6, activation="softmax")
    ])

    optimizer = Adam(learning_rate = 3e-4)
    model.compile(
        optimizer = optimizer,
        loss = "sparse_categorical_crossentropy",
        metrics = ["accuracy"]
    )
    return model

In [ ]:
model = build_dnn(X_train_scaled.shape)

In [ ]:
model.summary()

In [ ]:
from sklearn.utils import class_weight
classes=np.unique(y_train)
weights = class_weight.compute_class_weight('balanced', classes=classes, y=y_train)
class_weights = dict(zip(classes,weights))

In [ ]:
history = model.fit(
    X_train_scaled,
    y_train,
    validation_data = (X_val_scaled, y_val),
    epochs = 50,
    batch_size=128,
    callbacks =[
        EarlyStopping(
            monitor="val_loss",
            patience = 5,
            restore_best_weights = True
        ),
        ReduceLROnPlateau(
            monitor = "val_loss",
            factor = 0.2,
            patience =3,
            min_lr = 1e-5
        )
        
    ],
    class_weight = class_weights
)

In [ ]:
model.save("model.keras")

In [ ]:
# model = load_model("model_0.8f1.keras")

## Model Evalutation

In [ ]:
from sklearn.metrics import classification_report, f1_score, confusion_matrix

In [ ]:
X_test_scaled = selector.transform(X_test_scaled)

In [ ]:
def plot_metrics(history):
    
    train_loss = history['loss']
    val_loss = history['val_loss']
    train_acc = history['accuracy']
    val_acc = history['val_accuracy']

    # Accuracy
    plt.figure()
    plt.plot(train_acc, label='Training Accuracy')
    plt.plot(val_acc, label='Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.show()

    # Loss
    plt.figure()
    plt.plot(train_loss, label='Training Loss')
    plt.plot(val_loss, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

In [ ]:
plot_metrics(history.history)

In [ ]:
y_pred_probs = model.predict(X_test_scaled)
y_pred_classes = np.argmax(y_pred_probs, axis = 1)

In [ ]:
def plot_confusion_matrix(y_true, y_pred, classes):
    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    print(cm)
    # Compute normalized version (percentages)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    fig, ax = plt.subplots(1, 2, figsize=(20, 8))

    # Plot Raw Counts
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax[0],
                xticklabels=classes, yticklabels=classes)
    ax[0].set_title('Confusion Matrix (Counts)')
    ax[0].set_ylabel('True Label')
    ax[0].set_xlabel('Predicted Label')

    # Plot Normalized
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens', ax=ax[1],
                xticklabels=classes, yticklabels=classes)
    ax[1].set_title('Confusion Matrix (Normalized)')
    ax[1].set_ylabel('True Label')
    ax[1].set_xlabel('Predicted Label')

    plt.tight_layout()
    plt.show()

In [ ]:
plot_confusion_matrix(y_test, y_pred_classes, encoder.classes_)

In [ ]:
print(classification_report(y_test, y_pred_classes, digits=4, target_names=encoder.classes_))

In [ ]:
f1_score(y_test, y_pred_classes, average="macro")

In [ ]:
loss, accuracy = model.evaluate(X_test_scaled, y_test)
print(f"Test accuracy: {accuracy:.4f}")